# 15.3 Declaring network models by node and arc

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

AMPL's algebraic notation has great power to express a variety of network linear programs,
but the resulting constraint expressions are often not as natural as we would like.
While the idea of constraining "flow out minus flow in" at each `node` is easy to describe
and understand, the corresponding algebraic constraints tend to involve terms like

```ampl
sum {(i,k) in LINKS} Ship[i,k]
```

that are not so quickly comprehended. The more complex and realistic the network, the
worse the problem. Indeed, it can be hard to tell whether a model's algebraic constraints
represent a valid collection of flow balances on a network, and consequently whether specialized
network optimization software (described later in this chapter) can be used.

Algebraic formulations of network flows tend to be problematical because they are
constructed explicitly in terms of variables and constraints, while the nodes and arcs are
merely implicit in the way that the constraints are structured. People prefer to approach
network flow problems in the opposite way. They imagine giving an explicit definition
of nodes and arcs, from which flow variables and balance constraints implicitly arise. To
deal with this situation, AMPL provides an alternative that allows network concepts to be
declared directly in a model.
The network extensions to AMPL include two new kinds of declaration, `node` and
`arc`, that take the place of the `subject to` and `var` declarations in an algebraic constraint
formulation. The `node` declarations name the nodes of a network, and characterize
the flow balance constraints at the nodes. The `arc` declarations name and define the
arcs, by specifying the nodes that arcs connect, and by providing optional information
such as bounds and costs that are associated with arcs.

This section introduces `node` and `arc` by showing how they permit various examples
from earlier in this chapter to be reformulated conveniently. The following section presents
the rules for these declarations more systematically.

**A general transshipment model**

In rewriting the model of [Figure 15-2a](./15_1_minimumcost_transshipment_models.ipynb#fig-15-2a) using `node` and `arc`, we can retain all of the
set and `param` declarations and associated data. The changes affect only the three declarations
— minimize, `var`, and `subject to` — that define the linear program.

There is a `node` in the network for every member of the set CITIES. Using a `node`
declaration, we can say this directly:

```ampl
node Balance {k in CITIES}: net_in = demand[k] - supply[k];
```

The keyword `net_in` stands for "net input", that is, the flow in minus the flow out, so
this declaration says that net flow in must equal net demand at each `node` Balance[k].
Thus it says the same thing as the constraint named Balance[k] in the algebraic version,
except that it uses the concise term `net_in` in place of the lengthy expression

```ampl
sum {(i,k) in LINKS} Ship[i,k] - sum {(k,j) in LINKS} Ship[k,j]
```

Indeed, the syntax of `subject to` and `node` are practically the same except for the way
that the conservation-of-flow constraint is stated. (The keyword `net_out` may also be
used to stand for flow out minus flow in, so that we could have written `net_out` =
supply[k] - demand[k].)
There is an `arc` in the network for every pair in the set LINKS. This too can be said
directly, using an `arc` declaration:

```ampl
arc Ship {(i,j) in LINKS} >= 0, <= capacity[i,j],
       from Balance[i], to Balance[j], obj Total_Cost cost[i,j];
```

An `arc` Ship[i,j] is defined for each pair in LINKS, with bounds of 0 and
capacity[i,j] on its flow; to this extent, the `arc` and `var` declarations are the
same. The `arc` declaration contains additional phrases, however, to say that the `arc` runs
from the `node` named Balance[i] to the `node` named Balance[j], with a linear
coefficient of cost[i,j] in the objective function named Total_Cost. These
phrases use the keywords from, to, and obj.

Since the information about the objective function is included in the `arc` declaration,
it is not needed in the minimize declaration, which reduces to:

```ampl
minimize Total_Cost;
```

```ampl
set CITIES;
set LINKS within (CITIES cross CITIES);
param supply {CITIES} >= 0;        # amounts available at cities
param demand {CITIES} >= 0;        # amounts required at cities
       check: sum {i in CITIES} supply[i] = sum {j in CITIES} demand[j];
param cost {LINKS} >= 0;           # shipment costs/1000 packages
param capacity {LINKS} >= 0;       # max packages that can be shipped
minimize Total_Cost;
node Balance {k in CITIES}: net_in = demand[k] - supply[k];
arc Ship {(i,j) in LINKS} >= 0, <= capacity[i,j],
from Balance[i], to Balance[j], obj Total_Cost cost[i,j];
```

**[Figure 15-10](./15_3_declaring_network_models_by_node_and_arc.ipynb#fig-15-10):** General transshipment model with `node` and `arc` (net1node.mod).

The whole model is shown in [Figure 15-10](./15_3_declaring_network_models_by_node_and_arc.ipynb#fig-15-10).

As this description suggests, `arc` and `node` take the place of `var` and `subject to`,
respectively. In fact AMPL treats an `arc` declaration as a definition of variables, so that
you would still say display Ship to look at the optimal flows in the network model of
[Figure 15-10](./15_3_declaring_network_models_by_node_and_arc.ipynb#fig-15-10); it treats a `node` declaration as a definition of constraints. The difference is
that `node` and `arc` present the model in a way that corresponds more directly to its
appearance in a network diagram. The description of the nodes always comes first, followed
by a description of how the arcs connect the nodes.

**A specialized transshipment model**

The `node` and `arc` declarations make it easy to define a linear program for a network
that has several different kinds of nodes and arcs. For an example we return to the specialized
model of [Figure 15-4a](./15_1_minimumcost_transshipment_models.ipynb#fig-15-4a).

The network has a plant `node`, a distribution center `node` for each member of
D_CITY, and a warehouse `node` for each member of W_CITY. Thus the model requires
three `node` declarations:

```ampl
node Plant: net_out = p_supply;
node Dist {i in D_CITY};
node Whse {j in W_CITY}: net_in = w_demand[j];
```

The balance conditions say that flow out of `node` Plant must be p_supply, while flow
into `node` Whse[j] is w_demand[j]. (The network has no arcs into the plant or out
of the warehouses, so `net_out` and `net_in` are just the flow out and flow in, respectively.)
The conditions at `node` Dist[i] could be written either `net_in` = 0 or
`net_out` = 0, but since these are assumed by default we need not specify any condition
at all.

```ampl
set D_CITY;
set W_CITY;
set DW_LINKS within (D_CITY cross W_CITY);
param p_supply >= 0;                  # amount available at plant
param w_demand {W_CITY} >= 0;         # amounts required at warehouses
              check: p_supply = sum {j in W_CITY} w_demand[j];
param pd_cost {D_CITY} >= 0;   # shipment costs/1000 packages
param dw_cost {DW_LINKS} >= 0;
param pd_cap {D_CITY} >= 0;           # max packages that can be shipped
param dw_cap {DW_LINKS} >= 0;
minimize Total_Cost;
node Plant: net_out = p_supply;
node Dist {i in D_CITY};
node Whse {j in W_CITY}: net_in = w_demand[j];
arc PD_Ship {i in D_CITY} >= 0, <= pd_cap[i],
       from Plant, to Dist[i], obj Total_Cost pd_cost[i];
arc DW_Ship {(i,j) in DW_LINKS} >= 0, <= dw_cap[i,j],
       from Dist[i], to Whse[j], obj Total_Cost dw_cost[i,j];
```

**Figure 15-11:** Specialized transshipment model with `node` and `arc` (net3node.mod).

This network has two kinds of arcs. There is an `arc` from the plant to each member of
D_CITY, which can be declared by:

```ampl
arc PD_Ship {i in D_CITY} >= 0, <= pd_cap[i],
       from Plant, to Dist[i], obj Total_Cost pd_cost[i];
```

And there is an `arc` from distribution center `i` to warehouse `j` for each pair (i,j) in
DW_LINKS:

```ampl
arc DW_Ship {(i,j) in DW_LINKS} >= 0, <= dw_cap[i,j],
       from Dist[i], to Whse[j], obj Total_Cost dw_cost[i,j];
```

The `arc` declarations specify the relevant bounds and objective coefficients, as in our
previous example. The whole model is shown in Figure 15-11.

**Variations on transshipment models**

The balance conditions in `node` declarations may be inequalities, like ordinary algebraic
balance constraints. If production at the plant can sometimes exceed total demand
at the warehouses, it would be appropriate to give the condition in the declaration of `node`
Plant as `net_out` <= p_supply.

An `arc` declaration can specify losses in transit by adding a factor at the end of the
to phrase:

```ampl
arc PD_Ship {i in D_CITY} >= 0, <= pd_cap[i],
       from Plant, to Dist[i] 1-pd_loss[i],
       obj Total_Cost pd_cost[i];
```

This is interpreted as saying that PD_Ship[i] is the number of packages that leave
`node` Plant, but (1-pd_loss[i]) * PD_Ship[i] is the number that enter `node`
Dist[i].

The same option can be used to specify conversions. To use our previous example, if
shipments are measured in thousands of packages but demands are measured in cartons,
the arcs from distribution centers to warehouses should be declared as:

```ampl
arc DW_Ship {(i,j) in DW_LINKS} >= 0, <= dw_cap[i,j],
       from Dist[i], to Whse[j] (1000/ppc),
       obj Total_Cost dw_cost[i,j];
```

If the shipments to warehouses are also measured in cartons, the factor should be applied
at the distribution center:

```ampl
arc DW_Ship {(i,j) in DW_LINKS} >= 0, <= dw_cap[i,j],
       from Dist[i] (ppc/1000), to Whse[j],
       obj Total_Cost dw_cost[i,j];
```

A loss factor could also be applied to the to phrase in these examples.

**Maximum flow models**

In the diagram of [Figure 15-5](./15_2_other_network_models.ipynb#fig-15-5) that we have used to illustrate the maximum flow problem,
there are three kinds of intersections represented by nodes: the one where traffic
enters, the one where traffic leaves, and the others where traffic flow is conserved. Thus
a model of the network could have three corresponding `node` declarations:

```ampl
node Entr_Int: net_out >= 0;
node Exit_Int: net_in >= 0;
node Intersection {k in INTER diff {entr,exit}};
```

The condition `net_out` >= 0 implies that the flow out of `node` Entr_Int may be any
amount at all; this is the proper condition, since there is no balance constraint on the
entrance `node`. An analogous comment applies to the condition for `node` Exit_Int.

There is one `arc` in this network for each pair (i,j) in the set ROADS. Thus the declaration
should look something like this:

```ampl
arc Traff {(i,j) in ROADS} >= 0, <= cap[i,j],  # NOT RIGHT
       from Intersection[i], to Intersection[j],
       obj Entering_Traff (if i = entr then 1);
```

Since the aim is to maximize the total traffic leaving the entrance `node`, the `arc` is given a
coefficient of 1 in the objective if and only if `i` takes the value entr. When `i` does take
this value, however, the `arc` is specified to be from Intersection[entr], a `node`
that does not exist; the `arc` should rather be from `node` Entr_Int. Similarly, when `j`
takes the value exit, the `arc` should not be to Intersection[exit], but to
Exit_Int. AMPL will catch these errors and issue a message naming one of the nonexistent
nodes that has been referenced.
It might seem reasonable to use an if-then-else to get around this problem, in the
following way:

```ampl
arc Traff {(i,j) in ROADS} >= 0, <= cap[i,j], # SYNTAX ERROR
       from (if i = entr then Entr_Int else Intersection[i]),
       to   (if j = exit then Exit_Int else Intersection[j]),
       obj Entering_Traff (if i = entr then 1);
```

However, the if-then-else construct in AMPL does not apply to model components
such as Entr_Int and Intersection[i]; this version will be rejected as a syntax
error. Instead you need to use from and to phrases qualified by indexing expressions:

```ampl
arc Traff {(i,j) in ROADS} >= 0, <= cap[i,j],
       from {if i = entr} Entr_Int,
       from {if i <> entr} Intersection[i],
       to   {if j = exit} Exit_Int,
       to   {if j <> exit} Intersection[j],
       obj Entering_Traff (if i = entr then 1);
```

The special indexing expression beginning with if works much the same way here as it
does for constraints (Section 8.4); the from or to phrase is processed if the condition
following if is true. Thus Traff[i,j] is declared to be from Entr_Int if `i` equals
entr, and to be from Intersection[i] if `i` is not equal to entr, which is what we
intend.

As an alternative, we can combine the declarations of the three different kinds of
nodes into one. Observing that `net_out` is positive or zero for Entr_Int, negative or
zero for Exit_Int, and zero for all other nodes Intersection[i], we can declare:

```ampl
node Intersection {k in INTER}:
       (if k = exit then -Infinity)
       <= net_out <= (if k = entr then Infinity);
```

The nodes that were formerly declared as Entr_Int and Exit_Int are now just
Intersection[entr] and Intersection[exit], and consequently the `arc` declaration
that we previously marked "not right" now works just fine. The choice between
this version and the previous one is entirely a matter of convenience and taste.
(Infinity is a predefined AMPL parameter that may be used to specify any "infinitely
large" bound; its technical definition is given in Section A.7.2.)

Arguably the AMPL formulation that is most convenient and appealing is neither of
the above, but rather comes from interpreting the network diagram of [Figure 15-5](./15_2_other_network_models.ipynb#fig-15-5) in a
slightly different way. Suppose that we view the arrows into the entrance `node` and out of
the exit `node` as representing additional arcs, which happen to be adjacent to only one
`node` rather than two. Then flow in equals flow out at every intersection, and the `node`
declaration simplifies to:

```ampl
node Intersection {k in INTER};
```

```ampl
set INTER;                       # intersections
param entr symbolic in INTER;               # entrance to road network
param exit symbolic in INTER, <> entr;      # exit from road network
set ROADS within (INTER diff {exit}) cross (INTER diff {entr});
param cap {ROADS} >= 0;                     # capacities of roads
node Intersection {k in INTER};
arc Traff_In >= 0, to Intersection[entr];
arc Traff_Out >= 0, from Intersection[exit];
arc Traff {(i,j) in ROADS} >= 0, <= cap[i,j],
from Intersection[i], to Intersection[j];
maximize Entering_Traff: Traff_In;
data;
set INTER := a b c d e f g ;
param entr := a ;
param exit := g ;
param:   ROADS:  cap :=
         a b     50,   a     c  100
         b d     40,   b     e   20
         c d     60,   c     f   20
         d e     50,   d     f   60
         e g     70,   f     g   70 ;
```

**Figure 15-12:** Maximum flow model with `node` and `arc` (netmax3.mod).

The two arcs "hanging" at the entrance and exit are defined in the obvious way, but
include only a to or a from phrase:

```ampl
arc Traff_In >= 0, to Intersection[entr];
arc Traff_Out >= 0, from Intersection[exit];
```

The arcs that represent roads within the network are declared as before:

```ampl
arc Traff {(i,j) in ROADS} >= 0, <= cap[i,j],
       from Intersection[i], to Intersection[j];
```

When the model is represented in this way, the objective is to maximize Traff_In (or
equivalently Traff_Out). We could do this by adding an obj phrase to the `arc` declaration
for Traff_In, but in this case it is perhaps clearer to define the objective algebraically:

```ampl
maximize Entering_Traff: Traff_In;
```

This version is shown in full in Figure 15-12.